In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
from torch.utils.data import DataLoader
import pandas as pd 
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    logging
)
from utils.evaluation import * 
from utils.preprocessing import prepare_dataset, tokenize_function

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

logging.set_verbosity_error()

## Preprocessing

In [4]:
df_train = pd.read_csv("lemmatized\\train_lemmatized.csv")
df_val = pd.read_csv("lemmatized\\validation_lemmatized.csv")
df_test = pd.read_csv("lemmatized\\test_lemmatized.csv")
df_expert_45 = pd.read_csv("lemmatized\\expert_core_45_lemmatized.csv")
df_expert_90 = pd.read_csv("lemmatized\\expert_core_90_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized\\expert_core_180_lemmatized.csv")

dataset_train = prepare_dataset(df_train)
dataset_val = prepare_dataset(df_val)
dataset_test = prepare_dataset(df_test)
dataset_expert_45 = prepare_dataset(df_expert_45)
dataset_expert_90 = prepare_dataset(df_expert_90)
dataset_expert_180 = prepare_dataset(df_expert_180)

map_func = lambda examples: tokenize_function(examples, tokenizer)
tokenized_train = dataset_train.map(map_func, remove_columns=['text'], batched=True)
tokenized_val = dataset_val.map(map_func, remove_columns=['text'], batched=True)
tokenized_test = dataset_test.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_45 = dataset_expert_45.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_90 = dataset_expert_90.map(map_func, remove_columns=['text'], batched=True)
tokenized_expert_180 = dataset_expert_180.map(map_func, remove_columns=['text'], batched=True)

Map: 100%|██████████| 180/180 [00:00<00:00, 23497.50 examples/s]


## Baseline

In [5]:
model_zeroshot = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    problem_type="single_label_classification", 
    output_loading_info=False
).to(device)

model_zeroshot.eval()
test_dataset = tokenized_test.with_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

training_args = TrainingArguments(
    per_device_eval_batch_size=32,
    report_to="none",
    disable_tqdm=True
)

trainer_zeroshot = Trainer(
    model=model_zeroshot,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1235.97it/s, Materializing param=bert.pooler.dense.weight]                              


In [7]:
res_zeroshot = evaluate_trainer(trainer_zeroshot, test_dataset, "Base Model")

{'eval_loss': '1.101', 'eval_model_preparation_time': '0', 'eval_accuracy': '0.3372', 'eval_f1_macro': '0.287', 'eval_runtime': '2.333', 'eval_samples_per_second': '6429', 'eval_steps_per_second': '201'}

Результаты модели: Base Model
   Accuracy:     0.3372
   F1 Macro:     0.2870


## Training

In [8]:
def train_model(train_ds, eval_ds, name, epochs, batch_size, lr):
    print(f"\n{'='*50}")
    print(f"Обучение: {name}")
    print(f"{'='*50}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=3, 
        problem_type="single_label_classification", 
        ignore_mismatched_sizes=True
    ).to(device)

    args = TrainingArguments(
        eval_strategy="epoch" if eval_ds else "no",
        save_strategy="epoch" if eval_ds else "no",
        logging_steps=50,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        load_best_model_at_end=True if eval_ds else False,
        metric_for_best_model="f1_macro" if eval_ds else None,
        report_to="none",
        fp16=True if device == "cuda" else False,
        seed=42,
        disable_tqdm=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer

In [9]:
trainer_full = train_model(
    tokenized_train, tokenized_val, "full_supervised",
    epochs=3, batch_size=128, lr=2e-5
)


Обучение: full_supervised


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1635.86it/s, Materializing param=bert.pooler.dense.weight]                              


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.632794,0.625161,0.730067,0.730860
2,0.606866,0.598953,0.741133,0.744590
3,0.570820,0.588812,0.744733,0.746768


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.74it/s]


In [10]:
trainer_expert_45 = train_model(
    tokenized_expert_45, None, "expert_few_shot",
    epochs=50, batch_size=5, lr=2e-4
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1324.28it/s, Materializing param=bert.pooler.dense.weight]                              


Step,Training Loss
50,0.469122
100,0.006967
150,0.002987
200,0.001993
250,0.001573
300,0.001285
350,0.001109
400,0.001002
450,0.001012


In [11]:
trainer_expert_90 = train_model(
    tokenized_expert_90, None, "expert_few_shot",
    epochs=50, batch_size=10, lr=1e-4
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 909.34it/s, Materializing param=bert.pooler.dense.weight]                              


Step,Training Loss
50,0.634983
100,0.047319
150,0.009669
200,0.005820
250,0.004331
300,0.003566
350,0.003130
400,0.002813
450,0.002731


In [12]:
trainer_expert_180 = train_model(
    tokenized_expert_180, None, "expert_few_shot",
    epochs=38, batch_size=15, lr=2e-5
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1377.33it/s, Materializing param=bert.pooler.dense.weight]                              


Step,Training Loss
50,1.028611
100,0.819336
150,0.570818
200,0.399399
250,0.283830
300,0.200002
350,0.158592
400,0.136857
450,0.122516


## Evaluation

In [13]:
res_full = evaluate_trainer(trainer_full, tokenized_test, "Full Supervised (~42k примеров)")


Результаты модели: Full Supervised (~42k примеров)
   Accuracy:     0.7419
   F1 Macro:     0.7436


In [14]:
res_expert_45 = evaluate_trainer(trainer_expert_45, tokenized_test, "Few-Shot (45 примеров)")


Результаты модели: Few-Shot (45 примеров)
   Accuracy:     0.5458
   F1 Macro:     0.5479


In [15]:
res_expert_90 = evaluate_trainer(trainer_expert_90, tokenized_test, "Few-Shot (90 примеров)")


Результаты модели: Few-Shot (90 примеров)
   Accuracy:     0.5667
   F1 Macro:     0.5707


In [16]:
res_expert_180 = evaluate_trainer(trainer_expert_180, tokenized_test, "Few-Shot (180 примеров)")


Результаты модели: Few-Shot (180 примеров)
   Accuracy:     0.6016
   F1 Macro:     0.5995
